In [ ]:
import pandas as pd


spotify_df = pd.read_csv('data/raw_spotify_data.csv')

weather_df = pd.read_csv('data/api_weather_data.csv') 
audio_df = pd.read_csv('data/api_audio_features.csv')

# Clean Dates to ensure perfect merging
spotify_df['date'] = pd.to_datetime(spotify_df['date'])
weather_df['date'] = pd.to_datetime(weather_df['date'])

# Merge Spotify data with Last.fm Audio Features
music_df = pd.merge(spotify_df, audio_df, on=['track_name', 'artist_name'], how='left')

# Aggregate daily music metrics per country

daily_music_mood = music_df.groupby(['region', 'date'])['valence'].mean().reset_index()

# Merge aggregated music mood with daily weather data
final_df = pd.merge(daily_music_mood, weather_df, on=['region', 'date'], how='inner')

# Create the categorical 'Rain' column for T-Tests (Rain > 0mm = Yes, else No)
final_df['is_raining'] = final_df['precipitation_sum'].apply(lambda x: 'Yes' if x > 0 else 'No')

# Save the final file for Tableau and Statistical Analysis
final_df.to_csv('data/enriched_music_weather.csv', index=False)
print("Data merged and saved successfully!")